## Three Parts of this lab

### Part1: A simple "Agent" and "Agent Loop"
- Basically an LLM Call. We'll add tracing and streaming to the mix

### Part2: Adding Tool

A familiar one

### Part3: Adding Memory

So that different agent calls know about each other

In [34]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent 
from openai import OpenAI, AsyncOpenAI


from agents import Agent, Runner, trace, function_tool, SQLiteSession, set_default_openai_client, set_default_openai_api, set_trace_processors
from langsmith.integrations.openai_agents_sdk import OpenAIAgentsTracingProcessor

load_dotenv(override=True)

True

In [2]:
#set_tracing_disabled(True)

### Sidenote

The actual name of this framework on the official Python index pipi.org os openai-agents

So for your own projects in the future, you would do:
- pip install openai-agents or  uv add openai-agents
- followed by: from agents import Agent, Runner, trace

- Beware that doing a pip install agents would install something completely different - an older reinforcement learning library.

In [31]:
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
ollama = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# Tell SDK to use ollama client
set_default_openai_client(ollama)

# Tell the SDK to use chat completions
set_default_openai_api("chat_completions")



In [32]:
# Make an agent with name, instructions and model

agent = Agent(
    name="agent",
    instructions="You are a helpful assistant",
    model="qwen2.5:7b",
)



In [35]:
set_trace_processors([OpenAIAgentsTracingProcessor()])

In [36]:
result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")

d:\2026-courses\agent-track\.venv\Lib\site-packages\langsmith\_internal\_addressing.py:104: LangSmithBetaWarning: Addressing runs with `langsmith.address` is in beta and is enabled per workspace. A workspace without it rejects the run, so the trace is lost rather than falling back to a project. The behavior may change without notice.
  _warn_once(


In [6]:
result.final_output

"Sure! Here's a joke about autonomous AI agents:\n\nWhy did the autonomous AI agent refuse to cross the road?\n\nBecause it already had a neural net for every possible scenario, and walking on roads wasn't part of its training data!"

In [7]:
# The details of the llm calls
result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Sure! Here's a joke about autonomous AI agents:\n\nWhy did the autonomous AI agent refuse to cross the road?\n\nBecause it already had a neural net for every possible scenario, and walking on roads wasn't part of its training data!",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'qwen2.5:7b', 'response_id': 'chatcmpl-18'}}]

### Adding Observability with a trace


In [9]:
## Trace works only with openai API Key. With Ollama it silently does nothing. Let us use langsmith
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")

print(result.final_output)


[non-fatal] Tracing client error 401. Response data is redacted.


Sure, here's a joke about Autonomous AI Agents:

Why did the Autonomous AI Agent break up with the Smartwatch?

Because it wanted to live a life not dictated by notifications!


[non-fatal] Tracing client error 401. Response data is redacted.


## With Langsmith